<!-- MUTERBANDUNG_NOTEBOOK_STATUS_V1 -->
# Notebook Status - Wisata Training Utama

| Item | Isi |
|---|---|
| Status | UTAMA |
| Fungsi | Decision log utama untuk audit, readiness, dan keputusan dataset wisata. |
| Input utama | Dataset curated wisata dan artefak audit di `Wisata_Workspace/01_Dataset`. |
| Output utama | Keputusan bertahap terkait kesiapan data wisata. |
| Aturan pakai | Boleh dilanjutkan bertahap. Jangan loncat ke LLM sebelum fase readiness selesai. |
| Catatan | Pola penulisan wajib: tujuan kecil -> code/output -> keputusan singkat. |

---


# Wisata Training: Decision-Driven Data Preparation dan Audit

Notebook ini menjadi jalur utama untuk data wisata MuterBandung. Polanya dibuat seperti notebook penginapan: setiap tahap kecil menjalankan pengecekan, melihat output, lalu mencatat keputusan singkat.

Catatan: notebook lama disimpan sebagai arsip. Tahap awal ini belum masuk ke LLM atau perubahan model.


## Tahap 0 - Menetapkan Environment dan Artefak

Tujuan: memastikan notebook membaca folder wisata yang benar dan memakai dataset curated terbaru sebagai basis audit.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 90)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Wisata_Workspace").exists():
            return candidate
    raise FileNotFoundError("Wisata_Workspace tidak ditemukan dari cwd saat ini.")

PROJECT_ROOT = find_project_root()
WISATA_DIR = PROJECT_ROOT / "Wisata_Workspace"
CURATED_DIR = WISATA_DIR / "01_Dataset" / "3_Curated"
DOC_DIR = WISATA_DIR / "03_Dokumentasi"

paths = {
    "dataset_utama": CURATED_DIR / "DATABASE_WISATA_LABELED_V2_REVIEWED.csv",
    "validation_json": CURATED_DIR / "data_validation_results.json",
    "validation_report": DOC_DIR / "DATA_VALIDATION_REPORT_2026-05-25.md",
    "sentiment_score": WISATA_DIR / "01_Dataset" / "SENTIMENT_SCORES_PER_LOKASI.csv",
    "manual_batch3": CURATED_DIR / "manual_review_batch3_remaining_46_after_status_facility_2026-05-27.csv",
}

artifact_status = []
for name, path in paths.items():
    artifact_status.append({
        "artifact": name,
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        "modified_at": path.stat().st_mtime if path.exists() else None,
    })

artifact_status_df = pd.DataFrame(artifact_status)
display(artifact_status_df.drop(columns=["modified_at"]))
print("PROJECT_ROOT:", PROJECT_ROOT)


,artifact,relative_path,exists,size_kb
0,dataset_utama,Wisata_Workspace/01_Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED.csv,True,502.21
1,validation_json,Wisata_Workspace/01_Dataset/3_Curated/data_validation_results.json,True,13.01
2,validation_report,Wisata_Workspace/03_Dokumentasi/DATA_VALIDATION_REPORT_2026-05-25.md,True,6.43
3,sentiment_score,Wisata_Workspace/01_Dataset/SENTIMENT_SCORES_PER_LOKASI.csv,True,27.45
4,manual_batch3,Wisata_Workspace/01_Dataset/3_Curated/manual_review_batch3_remaining_46_after_status_f...,True,21.12


PROJECT_ROOT: D:\File\file\Fauzan Lubada\PIJAK


### Keputusan: environment wisata layak dipakai

Semua artefak awal yang dibutuhkan tersedia. Untuk tahap ini, dataset utama yang dipakai adalah `DATABASE_WISATA_LABELED_V2_REVIEWED.csv`, bukan notebook lama atau file Colab eksperimen.


## Fase A - Lineage Dataset Utama

Tujuan: melihat jumlah data, jumlah kolom, ID unik, dan status snapshot dataset yang akan dipakai untuk audit berikutnya.


In [2]:
df = pd.read_csv(paths["dataset_utama"], dtype=str, keep_default_na=False)
validation = json.loads(paths["validation_json"].read_text(encoding="utf-8"))
summary = validation.get("summary", {})

lineage_summary = pd.DataFrame([
    {"metric": "rows_csv", "value": len(df)},
    {"metric": "columns_csv", "value": len(df.columns)},
    {"metric": "unique_location_id_csv", "value": df["location_id"].nunique() if "location_id" in df.columns else None},
    {"metric": "rows_validation_json", "value": summary.get("row_count")},
    {"metric": "columns_validation_json", "value": summary.get("column_count")},
    {"metric": "unique_location_id_validation", "value": summary.get("unique_location_id_count")},
    {"metric": "gate_status", "value": summary.get("gate_status")},
    {"metric": "validated_at", "value": summary.get("validated_at")},
])

display(lineage_summary)

id_check = pd.DataFrame([
    {"check": "location_id kosong", "count": int((df.get("location_id", pd.Series(dtype=str)) == "").sum())},
    {"check": "location_id duplikat", "count": int(df["location_id"].duplicated().sum()) if "location_id" in df.columns else None},
    {"check": "location_name kosong", "count": int((df.get("location_name", pd.Series(dtype=str)) == "").sum())},
])
display(id_check)


,metric,value
0,rows_csv,234
1,columns_csv,87
2,unique_location_id_csv,234
3,rows_validation_json,234
4,columns_validation_json,87
5,unique_location_id_validation,234
6,gate_status,PASS_WITH_WARNINGS
7,validated_at,2026-06-02T03:47:41.925336Z


,check,count
0,location_id kosong,0
1,location_id duplikat,0
2,location_name kosong,0


### Keputusan: snapshot wisata diterima sebagai basis audit

Jumlah baris, kolom, dan ID unik konsisten dengan hasil validation JSON. Tidak ada indikasi awal bahwa snapshot ini salah file, jadi tahap berikutnya boleh memakai dataset ini sebagai basis audit wisata.


## Fase B - Status Dataset dan Kolom Inti

Tujuan: memastikan status destinasi dan kolom penting untuk rekomendasi wisata tersedia sebelum masuk ke audit warning.


In [3]:
status_counts = df["display_status"].value_counts(dropna=False).rename_axis("display_status").reset_index(name="count") if "display_status" in df.columns else pd.DataFrame()
display(status_counts)

core_columns = [
    "location_id", "location_name", "category", "final_primary_intent", "final_core_labels",
    "display_status", "is_active_verified", "latitude", "longitude", "coordinate_verified",
    "price_min", "price_max", "price_type", "jam_buka_weekday", "jam_tutup_weekday",
    "avg_rating", "total_ulasan", "sentiment_score", "sentimen_label_lokasi",
    "sentiment_available", "media_available", "media_image_url", "parking_verified",
    "toilet_verified", "mushola_verified", "child_friendly_verified", "night_verified",
]

core_audit = []
for col in core_columns:
    exists = col in df.columns
    if exists:
        empty_count = int((df[col].astype(str).str.strip() == "").sum())
        non_empty_count = int(len(df) - empty_count)
    else:
        empty_count = None
        non_empty_count = None
    core_audit.append({
        "column": col,
        "exists": exists,
        "non_empty_count": non_empty_count,
        "empty_count": empty_count,
    })

core_audit_df = pd.DataFrame(core_audit)
display(core_audit_df)


,display_status,count
0,active_candidate,209
1,exclude_scope,19
2,temporarily_hidden,3
3,status_uncertain,3


,column,exists,non_empty_count,empty_count
0,location_id,True,234,0
1,location_name,True,234,0
2,category,True,234,0
3,final_primary_intent,True,234,0
4,final_core_labels,True,234,0
5,display_status,True,234,0
6,is_active_verified,True,87,147
7,latitude,True,234,0
8,longitude,True,234,0
9,coordinate_verified,True,234,0


### Keputusan: kolom inti cukup untuk audit rekomendasi

Kolom utama untuk status, koordinat, harga, jam buka, rating, sentiment, media, dan fasilitas sudah tersedia. Karena masih ada nilai kosong pada beberapa kolom, tahap berikutnya tetap perlu membaca warning validation, bukan langsung menganggap data final bersih.


## Fase C - Warning Validation yang Masih Aktif

Tujuan: membaca hasil validation otomatis dan menentukan bagian mana yang harus dibawa ke audit lanjutan.


In [4]:
issues = pd.DataFrame(validation.get("issues", []))
if not issues.empty:
    issue_cols = ["severity", "code", "field", "count", "message"]
    issues_view = issues[issue_cols].sort_values(["severity", "count"], ascending=[True, False]).reset_index(drop=True)
    display(issues_view)
else:
    issues_view = pd.DataFrame(columns=["severity", "code", "field", "count", "message"])
    display(issues_view)

sample_rows = []
for issue in validation.get("issues", []):
    for row in issue.get("sample_rows", [])[:5]:
        sample_rows.append({
            "code": issue.get("code"),
            "field": issue.get("field"),
            "location_id": row.get("location_id"),
            "location_name": row.get("location_name"),
        })

sample_df = pd.DataFrame(sample_rows)
display(sample_df.head(30))

summary_view = pd.DataFrame([
    {"metric": "gate_status", "value": summary.get("gate_status")},
    {"metric": "error_count", "value": summary.get("error_count")},
    {"metric": "warning_count", "value": summary.get("warning_count")},
    {"metric": "info_count", "value": summary.get("info_count")},
    {"metric": "active_candidate_count", "value": summary.get("active_candidate_count")},
])
display(summary_view)


,severity,code,field,count,message
0,INFO,I_ACTIVE_WEBSITE_MISSING,media_website,139,Active candidates without official/reference website.
1,WARNING,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,147,Active candidates missing is_active_verified must not be described as verified.
2,WARNING,W_ACTIVE_FACILITY_VERIFICATION_SPARSE,facility_verified_flags,146,Active candidates with no verified facility flags require conservative LLM wording.
3,WARNING,W_ACTIVE_MEDIA_UNAVAILABLE,media_available,39,Active candidates without media need conservative UI/LLM presentation.
4,WARNING,W_ACTIVE_RATING_MISSING,avg_rating,16,Active candidates with missing avg_rating rely on runtime median fallback.
5,WARNING,W_ACTIVE_SENTIMENT_UNAVAILABLE,sentiment_available,16,Active candidates with unavailable sentiment need neutral/caveated LLM wording.
6,WARNING,W_ACTIVE_COORDINATE_UNVERIFIED,coordinate_verified,9,Active candidates with coordinate_verified=False should be treated carefully for dista...
7,WARNING,W_ZERO_PRICE_NOT_MARKED_FREE,"price_type,price_max",1,Rows with zero max price should usually be marked Gratis.
8,WARNING,W_OPEN_24H_FLAG_HOURS_MISMATCH,open_24h_verified,1,open_24h_verified=True should align with 00:00-23:59 weekday/weekend hours.


,code,field,location_id,location_name
0,W_ACTIVE_RATING_MISSING,avg_rating,LOC-029,Dusun Bambu
1,W_ACTIVE_RATING_MISSING,avg_rating,LOC-068,Pasar Cimindi
2,W_ACTIVE_RATING_MISSING,avg_rating,LOC-133,Punclut Bandung (Puncak Ciumbuleuit)
3,W_ACTIVE_RATING_MISSING,avg_rating,LOC-134,Bukit Bintang Bandung (Patahan Lembang)
4,W_ACTIVE_RATING_MISSING,avg_rating,LOC-152,Kebun Teh Rancabali
5,W_ZERO_PRICE_NOT_MARKED_FREE,"price_type,price_max",LOC-233,Kampung Wisata Pangjugjugan
6,W_OPEN_24H_FLAG_HOURS_MISMATCH,open_24h_verified,LOC-134,Bukit Bintang Bandung (Patahan Lembang)
7,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-001,23 Paskal Shopping Center
8,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-002,Alam Wisata Cimahi
9,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-003,Alun-Alun Kota Bandung


,metric,value
0,gate_status,PASS_WITH_WARNINGS
1,error_count,0
2,warning_count,375
3,info_count,139
4,active_candidate_count,209


### Keputusan: dataset bisa lanjut, tetapi masih perlu caveat

Gate validation berada di `PASS_WITH_WARNINGS`. Tidak ada error aktif, tetapi warning fasilitas, status aktif, media, rating, sentiment, dan koordinat perlu dibawa ke tahap audit berikutnya. Untuk LLM dan rekomendasi, sistem belum boleh membuat klaim yang terlalu pasti pada field yang belum verified.


## Fase D - Scope Destinasi dan Status Tayang

Tujuan: memisahkan destinasi yang layak tampil, ditahan, tidak masuk scope, atau masih perlu keputusan. Ini penting sebelum audit teknis, karena rekomendasi hanya boleh fokus ke data yang statusnya jelas.


In [5]:
status_scope = df.groupby(["display_status"], dropna=False).agg(
    total=("location_id", "count"),
    active_verified=("is_active_verified", lambda s: int((s.astype(str).str.lower() == "true").sum())),
    active_not_verified=("is_active_verified", lambda s: int((s.astype(str).str.lower() != "true").sum())),
).reset_index().sort_values("total", ascending=False)
display(status_scope)

status_detail_cols = ["location_id", "location_name", "category", "display_status", "curation_action", "is_active_verified", "curation_note"]
status_detail_cols = [c for c in status_detail_cols if c in df.columns]
status_detail = df.loc[df["display_status"].ne("active_candidate"), status_detail_cols].copy()
display(status_detail.sort_values(["display_status", "location_name"]).head(30))

active_df = df[df["display_status"].eq("active_candidate")].copy()
active_verification_summary = pd.DataFrame([
    {"metric": "active_candidate", "count": len(active_df)},
    {"metric": "active_verified_true", "count": int((active_df["is_active_verified"].astype(str).str.lower() == "true").sum())},
    {"metric": "active_verified_not_true", "count": int((active_df["is_active_verified"].astype(str).str.lower() != "true").sum())},
])
display(active_verification_summary)


,display_status,total,active_verified,active_not_verified
0,active_candidate,209,62,147
1,exclude_scope,19,0,19
2,status_uncertain,3,0,3
3,temporarily_hidden,3,0,3


,location_id,location_name,category,display_status,curation_action,is_active_verified,curation_note
3,LOC-004,Alun-Alun Sumedang,Taman Kota,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
8,LOC-009,Bendungan Jatigede,Wisata Alam,exclude_scope,remove,False,QA override: Jatigede/Sumedang destination is outside strict Bandung Raya scope.
14,LOC-015,Cadas Pangeran Sumedang,Tempat Sejarah,exclude_scope,remove,False,QA override: Sumedang historical site is outside strict Bandung Raya scope.
229,LOC-230,Desa Wisata Buricak Burinong,Desa Wisata,exclude_scope,remove,False,"Wisata valid, tetapi berada di Darmaraja, Sumedang/Waduk Jatigede, bukan Bandung Raya...."
36,LOC-037,Gn. Tampomas,Rekreasi Keluarga,exclude_scope,remove,False,QA override: Tampomas/Sumedang destination is outside strict Bandung Raya scope.
47,LOC-048,Kampung Karuhun Eco Green Park Sumedang,Rekreasi Keluarga,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
48,LOC-049,Kampung Toga Villa & Resto,Tempat Kuliner,exclude_scope,remove,False,QA override: Sumedang/Toga area destination is outside strict Bandung Raya scope.
232,LOC-233,Kampung Wisata Pangjugjugan,Rekreasi Alam,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
56,LOC-057,Masjid Agung Sumedang,Tempat Ibadah,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
58,LOC-059,Mata Air Cikandung,Wisata Alam,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.


,metric,count
0,active_candidate,209
1,active_verified_true,62
2,active_verified_not_true,147


### Keputusan: scope rekomendasi fokus ke active candidate

Destinasi dengan `active_candidate` menjadi basis utama rekomendasi. Baris `exclude_scope`, `temporarily_hidden`, dan `status_uncertain` tidak dibuang, tetapi tidak diperlakukan setara dengan destinasi aktif sampai ada keputusan yang lebih kuat.


## Fase E - Koordinat dan Distance Readiness

Tujuan: mengecek apakah koordinat cukup aman untuk klaim jarak, radius, dan fitur ?dekat saya?. Validasi ini hanya filter kasar, bukan batas administratif presisi.


In [6]:
coord_cols = ["location_id", "location_name", "display_status", "latitude", "longitude", "coordinate_verified", "distance_from_alun_alun_km", "coordinate_audit_reason"]
coord_cols = [c for c in coord_cols if c in df.columns]
coord_df = df[coord_cols].copy()

coord_df["lat_num"] = pd.to_numeric(coord_df.get("latitude", ""), errors="coerce")
coord_df["lon_num"] = pd.to_numeric(coord_df.get("longitude", ""), errors="coerce")
coord_df["distance_num"] = pd.to_numeric(coord_df.get("distance_from_alun_alun_km", ""), errors="coerce")
coord_df["coord_numeric_ok"] = coord_df["lat_num"].notna() & coord_df["lon_num"].notna()
coord_df["inside_rough_bandung_raya_box"] = coord_df["lat_num"].between(-7.35, -6.60) & coord_df["lon_num"].between(107.20, 108.15)
coord_df["coordinate_verified_bool"] = coord_df.get("coordinate_verified", "").astype(str).str.lower().eq("true")

coord_summary = pd.DataFrame([
    {"metric": "total_rows", "count": len(coord_df)},
    {"metric": "numeric_coordinate_ok", "count": int(coord_df["coord_numeric_ok"].sum())},
    {"metric": "missing_or_invalid_coordinate", "count": int((~coord_df["coord_numeric_ok"]).sum())},
    {"metric": "inside_rough_bandung_raya_box", "count": int(coord_df["inside_rough_bandung_raya_box"].sum())},
    {"metric": "outside_rough_bandung_raya_box", "count": int((coord_df["coord_numeric_ok"] & ~coord_df["inside_rough_bandung_raya_box"]).sum())},
    {"metric": "coordinate_verified_true", "count": int(coord_df["coordinate_verified_bool"].sum())},
    {"metric": "active_coordinate_unverified", "count": int((coord_df["display_status"].eq("active_candidate") & ~coord_df["coordinate_verified_bool"]).sum())},
    {"metric": "distance_available", "count": int(coord_df["distance_num"].notna().sum())},
])
display(coord_summary)

coord_attention = coord_df[
    coord_df["display_status"].eq("active_candidate") &
    ((~coord_df["coordinate_verified_bool"]) | (~coord_df["coord_numeric_ok"]) | (~coord_df["inside_rough_bandung_raya_box"]))
].copy()
attention_cols = [c for c in ["location_id", "location_name", "latitude", "longitude", "coordinate_verified", "distance_from_alun_alun_km", "coordinate_audit_reason"] if c in coord_attention.columns]
display(coord_attention[attention_cols].sort_values("location_name").head(30))

coordinate_audit_path = CURATED_DIR / "coordinate_audit.csv"
if coordinate_audit_path.exists():
    coordinate_audit_df = pd.read_csv(coordinate_audit_path, dtype=str, keep_default_na=False)
    display(coordinate_audit_df.head(20))
else:
    print("coordinate_audit.csv tidak ditemukan")


,metric,count
0,total_rows,234
1,numeric_coordinate_ok,234
2,missing_or_invalid_coordinate,0
3,inside_rough_bandung_raya_box,233
4,outside_rough_bandung_raya_box,1
5,coordinate_verified_true,220
6,active_coordinate_unverified,9
7,distance_available,234


,location_id,location_name,latitude,longitude,coordinate_verified,distance_from_alun_alun_km,coordinate_audit_reason
224,LOC-225,Curug Ceret Pangalengan,-7.3620716,107.5468974,True,49.41,
22,LOC-023,Curug Malela,-7.0115741,107.2056616,False,45.42,outside broad Bandung Raya coordinate bounds
154,LOC-155,Curug Panganten,-6.8435086,107.5639385,False,9.92,Pangalengan/Malabar coordinate too close to Bandung center: 9.9 km from Alun-Alun Bandung
43,LOC-044,Jans Park (Jatinangor National Park),-6.918795,107.769342,False,17.92,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 17.9...
137,LOC-138,Kebun Teh Sukawana,-6.89556,107.58361,False,3.9,Sukawana/Lembang coordinate too close to Bandung center: 3.9 km from Alun-Alun Bandung
165,LOC-166,Muara Rahong Hills,-6.9325,107.5625,False,5.06,Pangalengan/Malabar coordinate too close to Bandung center: 5.1 km from Alun-Alun Bandung
68,LOC-069,Pemandian Cipanas Cileungsing,-7.00583,107.51944,False,13.45,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 13.5...
161,LOC-162,Rumah Putih Cukul,-7.0315,107.7203,False,17.47,Pangalengan/Malabar coordinate too close to Bandung center: 17.5 km from Alun-Alun Ban...
157,LOC-158,Situ Ninah (Situ Datar),-7.03278,107.56611,False,13.15,Pangalengan/Malabar coordinate too close to Bandung center: 13.1 km from Alun-Alun Ban...
115,LOC-116,Wisata Kampoeng Ciherang,-6.8293636,107.7978925,False,23.44,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 23.4...


,location_name,category,display_status,curation_action,latitude,longitude,coordinate_verified,distance_from_alun_alun_km,coordinate_audit_reason
0,Curug Malela,Wisata Alam,active_candidate,keep,-7.0115741,107.2056616,False,45.42,outside broad Bandung Raya coordinate bounds
1,Gn. Tampomas,Rekreasi Keluarga,exclude_scope,remove,-6.8817995,107.5683276,False,6.17,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 6.2 ...
2,Jans Park (Jatinangor National Park),Rekreasi Keluarga,active_candidate,keep,-6.918795,107.769342,False,17.92,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 17.9...
3,Masjid Agung Sumedang,Tempat Ibadah,exclude_scope,remove,-6.907497,107.647137,False,4.7,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 4.7 ...
4,Pemandian Cipanas Cileungsing,Wisata Alam,active_candidate,keep,-7.00583,107.51944,False,13.45,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 13.5...
5,Wisata Kampoeng Ciherang,Rekreasi Keluarga,active_candidate,keep,-6.8293636,107.7978925,False,23.44,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 23.4...
6,Kebun Teh Sukawana,Wisata Alam,active_candidate,keep,-6.89556,107.58361,False,3.9,Sukawana/Lembang coordinate too close to Bandung center: 3.9 km from Alun-Alun Bandung
7,Curug Panganten,Wisata Alam,active_candidate,keep,-6.8435086,107.5639385,False,9.92,Pangalengan/Malabar coordinate too close to Bandung center: 9.9 km from Alun-Alun Bandung
8,Situ Ninah (Situ Datar),Wisata Alam,active_candidate,keep,-7.03278,107.56611,False,13.15,Pangalengan/Malabar coordinate too close to Bandung center: 13.1 km from Alun-Alun Ban...
9,Rumah Putih Cukul,Rekreasi Keluarga,active_candidate,keep,-7.0315,107.7203,False,17.47,Pangalengan/Malabar coordinate too close to Bandung center: 17.5 km from Alun-Alun Ban...


### Keputusan: fitur jarak bisa dipakai dengan guardrail

Koordinat sudah cukup untuk fitur radius dan jarak, tetapi baris yang belum `coordinate_verified=True` tetap harus diberi perlakuan hati-hati. Untuk lokasi yang masuk daftar perhatian, sistem boleh menghitung jarak, tetapi tidak boleh terlalu percaya diri pada klaim presisi.


## Fase F - Harga dan Jam Buka

Tujuan: mengecek apakah harga dan jam operasional cukup rapi untuk filter gratis/berbayar, range harga, buka sekarang, weekday, dan weekend.


In [7]:
price_time_cols = [
    "location_id", "location_name", "display_status", "price_min", "price_max", "price_type", "price_verified",
    "jam_buka", "jam_tutup", "jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend",
    "status_jam", "sumber_jam", "open_24h_verified", "night_verified", "catatan_jam",
]
price_time_cols = [c for c in price_time_cols if c in df.columns]
pt = df[price_time_cols].copy()

for col in ["price_min", "price_max"]:
    if col in pt.columns:
        pt[col + "_num"] = pd.to_numeric(pt[col].astype(str).str.replace(r"[^0-9.-]", "", regex=True), errors="coerce")

price_summary = pd.DataFrame([
    {"metric": "price_min_available", "count": int(pt.get("price_min_num", pd.Series(dtype=float)).notna().sum())},
    {"metric": "price_max_available", "count": int(pt.get("price_max_num", pd.Series(dtype=float)).notna().sum())},
    {"metric": "active_price_verified_true", "count": int((pt["display_status"].eq("active_candidate") & pt.get("price_verified", "").astype(str).str.lower().eq("true")).sum()) if "price_verified" in pt.columns else None},
    {"metric": "active_price_verified_not_true", "count": int((pt["display_status"].eq("active_candidate") & ~pt.get("price_verified", "").astype(str).str.lower().eq("true")).sum()) if "price_verified" in pt.columns else None},
])
display(price_summary)

if "price_type" in pt.columns:
    price_type_counts = pt.groupby(["display_status", "price_type"], dropna=False).size().reset_index(name="count").sort_values(["display_status", "count"], ascending=[True, False])
    display(price_type_counts)

zero_price_issue = pt[
    pt.get("price_max_num", pd.Series(index=pt.index, dtype=float)).eq(0) &
    ~pt.get("price_type", pd.Series(index=pt.index, dtype=str)).astype(str).str.lower().str.contains("gratis|free", na=False)
].copy()
zero_cols = [c for c in ["location_id", "location_name", "price_min", "price_max", "price_type", "price_verified"] if c in zero_price_issue.columns]
display(zero_price_issue[zero_cols].head(20))

hour_cols = ["jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend"]
for col in hour_cols:
    if col not in pt.columns:
        pt[col] = ""

pt["weekday_hours_complete"] = pt["jam_buka_weekday"].astype(str).str.strip().ne("") & pt["jam_tutup_weekday"].astype(str).str.strip().ne("")
pt["weekend_hours_complete"] = pt["jam_buka_weekend"].astype(str).str.strip().ne("") & pt["jam_tutup_weekend"].astype(str).str.strip().ne("")
pt["open_24h_bool"] = pt.get("open_24h_verified", "").astype(str).str.lower().eq("true") if "open_24h_verified" in pt.columns else False
pt["night_bool"] = pt.get("night_verified", "").astype(str).str.lower().eq("true") if "night_verified" in pt.columns else False

hour_summary = pd.DataFrame([
    {"metric": "weekday_hours_complete", "count": int(pt["weekday_hours_complete"].sum())},
    {"metric": "weekend_hours_complete", "count": int(pt["weekend_hours_complete"].sum())},
    {"metric": "active_weekday_hours_missing", "count": int((pt["display_status"].eq("active_candidate") & ~pt["weekday_hours_complete"]).sum())},
    {"metric": "active_weekend_hours_missing", "count": int((pt["display_status"].eq("active_candidate") & ~pt["weekend_hours_complete"]).sum())},
    {"metric": "open_24h_verified_true", "count": int(pt["open_24h_bool"].sum())},
    {"metric": "night_verified_true", "count": int(pt["night_bool"].sum())},
])
display(hour_summary)

hour_attention = pt[
    pt["display_status"].eq("active_candidate") &
    ((~pt["weekday_hours_complete"]) | (~pt["weekend_hours_complete"]) | pt["open_24h_bool"])
].copy()
hour_attention_cols = [c for c in ["location_id", "location_name", "jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend", "open_24h_verified", "night_verified", "status_jam", "catatan_jam"] if c in hour_attention.columns]
display(hour_attention[hour_attention_cols].head(30))


,metric,count
0,price_min_available,234
1,price_max_available,234
2,active_price_verified_true,209
3,active_price_verified_not_true,0


,display_status,price_type,count
2,active_candidate,Per Orang,157
1,active_candidate,Gratis,51
0,active_candidate,Berbayar,1
5,exclude_scope,Per Orang,14
4,exclude_scope,Gratis,4
3,exclude_scope,Berbayar,1
6,status_uncertain,Per Orang,3
7,temporarily_hidden,Per Orang,3


,location_id,location_name,price_min,price_max,price_type,price_verified
232,LOC-233,Kampung Wisata Pangjugjugan,0,0,Berbayar,False


,metric,count
0,weekday_hours_complete,233
1,weekend_hours_complete,233
2,active_weekday_hours_missing,0
3,active_weekend_hours_missing,0
4,open_24h_verified_true,15
5,night_verified_true,36


,location_id,location_name,jam_buka_weekday,jam_tutup_weekday,jam_buka_weekend,jam_tutup_weekend,open_24h_verified,night_verified,status_jam,catatan_jam
11,LOC-012,Bukit Moko Bandung,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
41,LOC-042,Gunung Putri Lembang,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
67,LOC-068,Pasar Cimindi,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
85,LOC-086,Taman Alun-Alun Kota Cimahi,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
88,LOC-089,Taman Film,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
91,LOC-092,Taman Jomblo,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
95,LOC-096,Taman Lansia,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
97,LOC-098,Taman Musik,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
98,LOC-099,Taman Superhero,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
133,LOC-134,Bukit Bintang Bandung (Patahan Lembang),06:00,17:00,06:00,17:00,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.


### Keputusan: filter harga dan jam buka bisa dipakai, tetapi tidak boleh dianggap sempurna

Kolom harga dan jam operasional sudah tersedia untuk filter dasar. Baris dengan harga nol yang tidak konsisten, jam kosong, atau `open_24h_verified` yang perlu dicek tetap masuk daftar perhatian agar filter ?gratis? dan ?buka sekarang? tidak memberi klaim keliru.


## Fase G - Fasilitas dan Verification Flags

Tujuan: mengecek fasilitas yang sudah terverifikasi. Nilai kosong atau `unknown` tidak boleh dianggap sama dengan fasilitas tidak tersedia.


In [8]:
facility_cols = [
    "parking_verified", "wheelchair_accessible_verified", "toilet_verified", "mushola_verified",
    "pet_friendly_verified", "child_friendly_verified", "indoor_verified", "safety_verified",
]
facility_cols = [c for c in facility_cols if c in df.columns]

facility_rows = []
for col in facility_cols:
    s = df[col].astype(str).str.strip().str.lower()
    active_s = df.loc[df["display_status"].eq("active_candidate"), col].astype(str).str.strip().str.lower()
    facility_rows.append({
        "flag": col,
        "true_all": int(s.eq("true").sum()),
        "false_all": int(s.eq("false").sum()),
        "empty_or_unknown_all": int((s.eq("") | s.eq("unknown") | s.eq("nan")).sum()),
        "true_active": int(active_s.eq("true").sum()),
        "false_active": int(active_s.eq("false").sum()),
        "empty_or_unknown_active": int((active_s.eq("") | active_s.eq("unknown") | active_s.eq("nan")).sum()),
    })
facility_summary = pd.DataFrame(facility_rows)
display(facility_summary)

active_facility = df[df["display_status"].eq("active_candidate")].copy()
for col in facility_cols:
    active_facility[col + "_bool"] = active_facility[col].astype(str).str.strip().str.lower().eq("true")
facility_bool_cols = [col + "_bool" for col in facility_cols]
active_facility["verified_facility_count"] = active_facility[facility_bool_cols].sum(axis=1) if facility_bool_cols else 0

facility_distribution = active_facility["verified_facility_count"].value_counts().rename_axis("verified_facility_count").reset_index(name="active_location_count").sort_values("verified_facility_count")
display(facility_distribution)

facility_sparse_cols = ["location_id", "location_name", "category", "display_status", "verified_facility_count", *facility_cols]
facility_sparse = active_facility[active_facility["verified_facility_count"].eq(0)][facility_sparse_cols].copy()
display(facility_sparse.sort_values("location_name").head(30))


,flag,true_all,false_all,empty_or_unknown_all,true_active,false_active,empty_or_unknown_active
0,parking_verified,58,176,0,58,151,0
1,wheelchair_accessible_verified,9,225,0,9,200,0
2,toilet_verified,55,179,0,55,154,0
3,mushola_verified,59,175,0,57,152,0
4,pet_friendly_verified,0,234,0,0,209,0
5,child_friendly_verified,76,158,0,66,143,0
6,indoor_verified,34,200,0,33,176,0
7,safety_verified,78,156,0,71,138,0


,verified_facility_count,active_location_count
0,0,83
1,1,43
2,2,27
5,3,11
4,4,13
3,5,27
6,6,3
7,7,2


,location_id,location_name,category,display_status,verified_facility_count,parking_verified,wheelchair_accessible_verified,toilet_verified,mushola_verified,pet_friendly_verified,child_friendly_verified,indoor_verified,safety_verified
10,LOC-011,Bukit Jamur Rancabolang Pasirjambu,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False
11,LOC-012,Bukit Moko Bandung,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False
12,LOC-013,Bukit Senyum,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False
13,LOC-014,Bumi Perkemahan Ranca Upas Bandung,Tempat Camping,active_candidate,0,False,False,False,False,False,False,False,False
130,LOC-131,Cikole Jayagiri Resort,Penginapan Wisata,active_candidate,0,False,False,False,False,False,False,False,False
184,LOC-185,Ciwangun Indah Camp,Tempat Camping,active_candidate,0,False,False,False,False,False,False,False,False
189,LOC-190,Curug Anom,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False
188,LOC-189,Curug Aseupan,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False
200,LOC-201,Curug Batu Templek,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False
187,LOC-188,Curug Bugbrug,Wisata Alam,active_candidate,0,False,False,False,False,False,False,False,False


### Keputusan: fasilitas dipakai sebagai sinyal positif, bukan klaim negatif

Fasilitas yang `true` boleh dipakai untuk filter dan alasan rekomendasi. Baris yang kosong atau belum verified tidak boleh otomatis disebut tidak punya fasilitas; cukup dianggap belum kuat datanya.


## Fase H - Media, Gambar, dan Website

Tujuan: mengecek kesiapan media untuk tampilan website. Media penting untuk UI, tetapi hasil match tetap perlu guardrail jika skornya lemah atau masuk review queue.


In [9]:
media_path = CURATED_DIR / "media_groundtruth_audit.csv"
media_queue_path = CURATED_DIR / "media_match_review_queue.csv"

media_df = pd.read_csv(media_path, dtype=str, keep_default_na=False) if media_path.exists() else pd.DataFrame()
media_queue_df = pd.read_csv(media_queue_path, dtype=str, keep_default_na=False) if media_queue_path.exists() else pd.DataFrame()

media_cols = ["location_id", "location_name", "display_status", "media_available", "media_image_url", "media_destination_url", "media_website", "media_source", "media_match_score", "media_audit_status", "media_audit_note"]
media_base = df[[c for c in media_cols if c in df.columns]].copy()

media_base["media_available_bool"] = media_base.get("media_available", "").astype(str).str.lower().eq("true")
media_base["image_url_available"] = media_base.get("media_image_url", "").astype(str).str.strip().ne("")
media_base["website_available"] = media_base.get("media_website", "").astype(str).str.strip().ne("")
media_base["media_match_score_num"] = pd.to_numeric(media_base.get("media_match_score", ""), errors="coerce")

media_summary = pd.DataFrame([
    {"metric": "rows_dataset", "count": len(media_base)},
    {"metric": "media_available_true", "count": int(media_base["media_available_bool"].sum())},
    {"metric": "image_url_available", "count": int(media_base["image_url_available"].sum())},
    {"metric": "website_available", "count": int(media_base["website_available"].sum())},
    {"metric": "active_media_unavailable", "count": int((media_base["display_status"].eq("active_candidate") & ~media_base["media_available_bool"]).sum())},
    {"metric": "media_groundtruth_audit_rows", "count": len(media_df)},
    {"metric": "media_match_review_queue_rows", "count": len(media_queue_df)},
])
display(media_summary)

if "media_audit_status" in media_base.columns:
    media_status_counts = media_base.groupby(["media_audit_status"], dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)
    display(media_status_counts)

score_summary = media_base["media_match_score_num"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).reset_index()
score_summary.columns = ["metric", "media_match_score"]
display(score_summary)

media_attention = media_base[
    media_base["display_status"].eq("active_candidate") &
    ((~media_base["media_available_bool"]) | (media_base["media_match_score_num"].fillna(0) < 0.75))
].copy()
attention_cols = [c for c in ["location_id", "location_name", "media_available", "media_match_score", "media_audit_status", "media_audit_note", "media_image_url"] if c in media_attention.columns]
display(media_attention[attention_cols].sort_values("location_name").head(30))

if not media_queue_df.empty:
    display(media_queue_df.head(20))


,metric,count
0,rows_dataset,234
1,media_available_true,183
2,image_url_available,183
3,website_available,75
4,active_media_unavailable,39
5,media_groundtruth_audit_rows,234
6,media_match_review_queue_rows,53


,media_audit_status,count
0,accepted,183
1,missing,51


,metric,media_match_score
0,count,234.00000
1,mean,0.92548
2,std,0.14586
3,min,0.44780
4,25%,1.00000
5,50%,1.00000
6,75%,1.00000
7,90%,1.00000
8,max,1.00000


,location_id,location_name,media_available,media_match_score,media_audit_status,media_audit_note,media_image_url
133,LOC-134,Bukit Bintang Bandung (Patahan Lembang),False,0.6102,missing,No reliable media match.,
172,LOC-173,Cakrawala Nature Sparkling Restaurant,False,0.6286,missing,No reliable media match.,
16,LOC-017,Cihampelas Walk,False,0.5185,missing,No reliable media match.,
179,LOC-180,Cimory Dairyland Lembang,False,0.5854,missing,No reliable media match.,
189,LOC-190,Curug Anom,False,0.8,missing,No reliable media match.,
200,LOC-201,Curug Batu Templek,False,0.7347,missing,No reliable media match.,
224,LOC-225,Curug Ceret Pangalengan,False,0.7273,missing,No reliable media match.,
186,LOC-187,Curug Layung,False,0.8,missing,No reliable media match.,
191,LOC-192,Curug Ngebul Gununghalu,False,0.6111,missing,No reliable media match.,
195,LOC-196,Curug Walanda Citatah,False,0.7647,missing,No reliable media match.,


,location_id,location_name,media_match_title,media_match_score,media_match_method,media_audit_status,media_audit_note
0,LOC-155,Curug Panganten,Jl. Curug Panganten,0.9091,manual_reject,missing,"Road/route result, not destination-level media."
1,LOC-037,Gn. Tampomas,Jl. Gn. Tampomas,0.88,manual_reject,missing,"Road result, not destination-level media."
2,LOC-176,Padepokan Dayang Sumbi,Penginapan Dayang Sumbi,0.8,missing,missing,No reliable media match.
3,LOC-177,Wahoo Waterworld,Wahoo Waterworld Bandung,0.8,missing,missing,No reliable media match.
4,LOC-182,Pine Forest Camp Lembang,Pine Forest Camp,0.8,missing,missing,No reliable media match.
5,LOC-187,Curug Layung,Curug Pelangi,0.8,missing,missing,No reliable media match.
6,LOC-190,Curug Anom,Curug Dago,0.8,missing,missing,No reliable media match.
7,LOC-228,Tanjung Duriat,Wisata Tanjung Duriat,0.8,missing,missing,No reliable media match.
8,LOC-159,Penangkaran Rusa Kertamanah,Penangkaran Rusa Ranca Upas,0.7778,missing,missing,No reliable media match.
9,LOC-196,Curug Walanda Citatah,Curug Walanda,0.7647,missing,missing,No reliable media match.


### Keputusan: media cukup untuk tampilan awal, tetapi belum semua destinasi kuat

Destinasi dengan media valid bisa ditampilkan lebih percaya diri. Destinasi tanpa media atau dengan match lemah tetap boleh ada di sistem, tetapi UI dan LLM sebaiknya tidak menonjolkan gambar sebagai bukti utama.


## Fase I - Sentiment Lineage dan Coverage

Tujuan: memastikan sentiment yang dipakai punya sumber yang jelas, jumlah lokasi tercakup cukup, dan baris kosong tidak dipaksa menjadi klaim positif atau negatif.


In [10]:
sentiment_path = WISATA_DIR / "01_Dataset" / "SENTIMENT_SCORES_PER_LOKASI.csv"
reviews_enriched_path = WISATA_DIR / "01_Dataset" / "MASTER_REVIEWS_ENRICHED.csv"
reviews_labeled_path = WISATA_DIR / "01_Dataset" / "MASTER_REVIEWS_LABELED.csv"

sentiment_df = pd.read_csv(sentiment_path, dtype=str, keep_default_na=False) if sentiment_path.exists() else pd.DataFrame()
reviews_enriched_df = pd.read_csv(reviews_enriched_path, dtype=str, keep_default_na=False) if reviews_enriched_path.exists() else pd.DataFrame()
reviews_labeled_df = pd.read_csv(reviews_labeled_path, dtype=str, keep_default_na=False) if reviews_labeled_path.exists() else pd.DataFrame()

sentiment_base = df[[c for c in ["location_id", "location_name", "display_status", "total_ulasan", "avg_rating", "sentiment_score", "avg_sentimen_skor", "sentimen_label_lokasi", "sentiment_model_source", "sentiment_model_version", "sentiment_available"] if c in df.columns]].copy()
sentiment_base["sentiment_available_bool"] = sentiment_base.get("sentiment_available", "").astype(str).str.lower().eq("true")
sentiment_base["sentiment_score_num"] = pd.to_numeric(sentiment_base.get("sentiment_score", ""), errors="coerce")
sentiment_base["total_ulasan_num"] = pd.to_numeric(sentiment_base.get("total_ulasan", ""), errors="coerce")

sentiment_summary = pd.DataFrame([
    {"metric": "dataset_locations", "count": len(sentiment_base)},
    {"metric": "sentiment_available_true", "count": int(sentiment_base["sentiment_available_bool"].sum())},
    {"metric": "active_sentiment_unavailable", "count": int((sentiment_base["display_status"].eq("active_candidate") & ~sentiment_base["sentiment_available_bool"]).sum())},
    {"metric": "sentiment_score_file_rows", "count": len(sentiment_df)},
    {"metric": "reviews_enriched_rows", "count": len(reviews_enriched_df)},
    {"metric": "reviews_labeled_rows", "count": len(reviews_labeled_df)},
])
display(sentiment_summary)

if "sentiment_model_source" in sentiment_base.columns:
    source_counts = sentiment_base.groupby(["sentiment_model_source", "sentiment_model_version"], dropna=False).size().reset_index(name="location_count").sort_values("location_count", ascending=False)
    display(source_counts)

label_counts = sentiment_base.groupby(["sentimen_label_lokasi"], dropna=False).size().reset_index(name="location_count").sort_values("location_count", ascending=False)
display(label_counts)

score_distribution = sentiment_base["sentiment_score_num"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).reset_index()
score_distribution.columns = ["metric", "sentiment_score"]
display(score_distribution)

low_confidence_sentiment = sentiment_base[
    sentiment_base["display_status"].eq("active_candidate") &
    ((~sentiment_base["sentiment_available_bool"]) | (sentiment_base["total_ulasan_num"].fillna(0) < 30))
].copy()
low_cols = [c for c in ["location_id", "location_name", "total_ulasan", "avg_rating", "sentiment_score", "sentimen_label_lokasi", "sentiment_model_source", "sentiment_available"] if c in low_confidence_sentiment.columns]
display(low_confidence_sentiment[low_cols].sort_values(["sentiment_available", "total_ulasan"], ascending=[True, True]).head(40))

adjusted_columns = [c for c in df.columns if "adjust" in c.lower() or "confidence" in c.lower()]
adjusted_status = pd.DataFrame({"column": adjusted_columns})
display(adjusted_status)


,metric,count
0,dataset_locations,234
1,sentiment_available_true,212
2,active_sentiment_unavailable,16
3,sentiment_score_file_rows,212
4,reviews_enriched_rows,34150
5,reviews_labeled_rows,34003


,sentiment_model_source,sentiment_model_version,location_count
0,tfidf_linearsvc,run_nlp_pipeline_v2,212
1,unavailable,,22


,sentimen_label_lokasi,location_count
4,Sangat Positif,180
3,Positif,27
0,,22
2,Netral,4
1,Negatif,1


,metric,sentiment_score
0,count,234.000000
1,mean,0.699707
2,std,0.291964
3,min,-0.307692
4,5%,0.000000
5,25%,0.616987
6,50%,0.795018
7,75%,0.894111
8,95%,0.975000
9,max,1.000000


,location_id,location_name,total_ulasan,avg_rating,sentiment_score,sentimen_label_lokasi,sentiment_model_source,sentiment_available
28,LOC-029,Dusun Bambu,0.0,,0.0,,unavailable,False
67,LOC-068,Pasar Cimindi,0.0,,0.0,,unavailable,False
132,LOC-133,Punclut Bandung (Puncak Ciumbuleuit),0.0,,0.0,,unavailable,False
133,LOC-134,Bukit Bintang Bandung (Patahan Lembang),0.0,,0.0,,unavailable,False
151,LOC-152,Kebun Teh Rancabali,0.0,,0.0,,unavailable,False
154,LOC-155,Curug Panganten,0.0,,0.0,,unavailable,False
161,LOC-162,Rumah Putih Cukul,0.0,,0.0,,unavailable,False
173,LOC-174,Lereng Anteng Panoramic Coffee,0.0,,0.0,,unavailable,False
175,LOC-176,Padepokan Dayang Sumbi,0.0,,0.0,,unavailable,False
191,LOC-192,Curug Ngebul Gununghalu,0.0,,0.0,,unavailable,False


,column
0,label_confidence
1,manual_review_confidence


### Keputusan: sentiment bisa dipakai untuk ranking, tetapi lineage harus jujur

Sentiment tersedia untuk mayoritas destinasi, namun baris tanpa sentiment atau dengan ulasan rendah harus diberi confidence lebih rendah. Jika kolom adjusted sentiment belum ada di dataset utama, tahap berikutnya perlu memastikan skornya sudah dikalibrasi sebelum dipakai sebagai sinyal utama ranking.


## Fase J - Label, Intent, Kategori, dan Filter Website

Tujuan: mengecek apakah label wisata sudah cukup untuk mendukung filter website. Fokusnya bukan membuat filter baru, tetapi melihat field mana yang aman dipakai user.


In [11]:
import ast
from collections import Counter

label_cols = [
    "location_id", "location_name", "display_status", "category", "subcategory",
    "primary_intent", "final_primary_intent", "core_labels", "final_core_labels",
    "secondary_labels", "final_secondary_labels", "avoid_labels", "final_avoid_labels",
    "label_confidence", "needs_manual_review", "review_status", "final_label_source",
]
label_cols = [c for c in label_cols if c in df.columns]
label_df = df[label_cols].copy()
active_label_df = label_df[label_df["display_status"].eq("active_candidate")].copy()

def normalize_list_value(value):
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    separators = [";", ",", "|"]
    for sep in separators:
        if sep in text:
            return [x.strip() for x in text.split(sep) if x.strip()]
    return [text]

category_counts = active_label_df["category"].value_counts(dropna=False).rename_axis("category").reset_index(name="active_count")
display(category_counts)

intent_counts = active_label_df["final_primary_intent"].replace("", "missing").value_counts(dropna=False).rename_axis("final_primary_intent").reset_index(name="active_count") if "final_primary_intent" in active_label_df.columns else pd.DataFrame()
display(intent_counts)

core_counter = Counter()
for value in active_label_df.get("final_core_labels", pd.Series(dtype=str)):
    core_counter.update(normalize_list_value(value))
core_label_counts = pd.DataFrame(core_counter.most_common(), columns=["final_core_label", "active_count"])
display(core_label_counts.head(30))

label_health = pd.DataFrame([
    {"metric": "active_candidate", "count": len(active_label_df)},
    {"metric": "missing_category", "count": int(active_label_df.get("category", pd.Series(dtype=str)).astype(str).str.strip().eq("").sum())},
    {"metric": "missing_final_primary_intent", "count": int(active_label_df.get("final_primary_intent", pd.Series(dtype=str)).astype(str).str.strip().eq("").sum())},
    {"metric": "missing_final_core_labels", "count": int(active_label_df.get("final_core_labels", pd.Series(dtype=str)).astype(str).str.strip().eq("").sum())},
    {"metric": "needs_manual_review_true", "count": int(active_label_df.get("needs_manual_review", pd.Series(dtype=str)).astype(str).str.lower().eq("true").sum())},
])
display(label_health)

if "label_confidence" in active_label_df.columns:
    active_label_df["label_confidence_num"] = pd.to_numeric(active_label_df["label_confidence"], errors="coerce")
    label_confidence_summary = active_label_df["label_confidence_num"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).reset_index()
    label_confidence_summary.columns = ["metric", "label_confidence"]
    display(label_confidence_summary)

filter_specs = [
    {"filter": "kategori wisata", "fields": ["category", "final_core_labels"], "logic": "non_empty", "recommendation": "Tampilkan sebagai filter utama"},
    {"filter": "tujuan kunjungan", "fields": ["final_primary_intent"], "logic": "non_empty", "recommendation": "Tampilkan sebagai filter utama"},
    {"filter": "gratis/berbayar", "fields": ["price_type", "price_min", "price_max"], "logic": "price_available", "recommendation": "Tampilkan sebagai filter utama"},
    {"filter": "buka sekarang", "fields": ["jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend"], "logic": "hours_complete", "recommendation": "Tampilkan dengan guardrail jam"},
    {"filter": "ramah anak", "fields": ["child_friendly_verified"], "logic": "true_count", "recommendation": "Tampilkan sebagai quick filter"},
    {"filter": "indoor", "fields": ["indoor_verified"], "logic": "true_count", "recommendation": "Tampilkan sebagai filter lanjutan"},
    {"filter": "malam hari", "fields": ["night_verified"], "logic": "true_count", "recommendation": "Tampilkan sebagai quick filter"},
    {"filter": "parkir", "fields": ["parking_verified"], "logic": "true_count", "recommendation": "Tampilkan sebagai filter fasilitas"},
    {"filter": "toilet", "fields": ["toilet_verified"], "logic": "true_count", "recommendation": "Tampilkan sebagai filter fasilitas"},
    {"filter": "mushola", "fields": ["mushola_verified"], "logic": "true_count", "recommendation": "Tampilkan sebagai filter fasilitas"},
    {"filter": "rating/sentiment", "fields": ["avg_rating", "sentiment_score", "sentiment_available"], "logic": "sentiment_available", "recommendation": "Tampilkan, tetapi baca confidence"},
    {"filter": "ada foto/media", "fields": ["media_available", "media_image_url"], "logic": "media_available", "recommendation": "Tampilkan sebagai quick filter"},
    {"filter": "jarak/radius", "fields": ["latitude", "longitude", "coordinate_verified"], "logic": "coordinate_numeric", "recommendation": "Tampilkan dengan guardrail koordinat"},
]

filter_rows = []
active_full = df[df["display_status"].eq("active_candidate")].copy()
for spec in filter_specs:
    fields = spec["fields"]
    missing_fields = [f for f in fields if f not in df.columns]
    if missing_fields:
        coverage = None
        supporting_rows = None
        status = "blocked_missing_field"
    else:
        if spec["logic"] == "non_empty":
            mask = pd.Series(True, index=active_full.index)
            for f in fields:
                mask &= active_full[f].astype(str).str.strip().ne("")
        elif spec["logic"] == "price_available":
            mask = active_full["price_type"].astype(str).str.strip().ne("") & active_full["price_min"].astype(str).str.strip().ne("") & active_full["price_max"].astype(str).str.strip().ne("")
        elif spec["logic"] == "hours_complete":
            mask = active_full["jam_buka_weekday"].astype(str).str.strip().ne("") & active_full["jam_tutup_weekday"].astype(str).str.strip().ne("") & active_full["jam_buka_weekend"].astype(str).str.strip().ne("") & active_full["jam_tutup_weekend"].astype(str).str.strip().ne("")
        elif spec["logic"] == "true_count":
            mask = active_full[fields[0]].astype(str).str.lower().eq("true")
        elif spec["logic"] == "sentiment_available":
            mask = active_full["sentiment_available"].astype(str).str.lower().eq("true") & active_full["sentiment_score"].astype(str).str.strip().ne("")
        elif spec["logic"] == "media_available":
            mask = active_full["media_available"].astype(str).str.lower().eq("true") & active_full["media_image_url"].astype(str).str.strip().ne("")
        elif spec["logic"] == "coordinate_numeric":
            mask = pd.to_numeric(active_full["latitude"], errors="coerce").notna() & pd.to_numeric(active_full["longitude"], errors="coerce").notna()
        else:
            mask = pd.Series(False, index=active_full.index)
        supporting_rows = int(mask.sum())
        coverage = round(supporting_rows / len(active_full) * 100, 1) if len(active_full) else 0
        if coverage >= 80:
            status = "strong_ready"
        elif coverage >= 30:
            status = "usable_with_guardrail"
        elif supporting_rows > 0:
            status = "limited_use"
        else:
            status = "not_ready"
    filter_rows.append({
        "filter": spec["filter"],
        "fields": ", ".join(fields),
        "supporting_active_rows": supporting_rows,
        "coverage_pct": coverage,
        "readiness_status": status,
        "recommendation": spec["recommendation"],
    })

filter_readiness_df = pd.DataFrame(filter_rows)
display(filter_readiness_df)

label_attention_cols = [c for c in ["location_id", "location_name", "category", "final_primary_intent", "final_core_labels", "label_confidence", "needs_manual_review", "review_status", "final_label_source"] if c in active_label_df.columns]
label_attention = active_label_df[
    active_label_df.get("final_primary_intent", pd.Series(index=active_label_df.index, dtype=str)).astype(str).str.strip().eq("") |
    active_label_df.get("final_core_labels", pd.Series(index=active_label_df.index, dtype=str)).astype(str).str.strip().eq("") |
    active_label_df.get("needs_manual_review", pd.Series(index=active_label_df.index, dtype=str)).astype(str).str.lower().eq("true")
].copy()
display(label_attention[label_attention_cols].head(40))


,category,active_count
0,Wisata Alam,77
1,Rekreasi Keluarga,35
2,Taman Kota,18
3,Tempat Belajar,14
4,Tempat Camping,10
5,Tempat Belanja,9
6,Tempat Sejarah,6
7,Wisata Satwa,6
8,Tempat Kuliner,6
9,Tempat Ibadah,5


,final_primary_intent,active_count
0,Alam,90
1,Keluarga,44
2,Santai/Healing,19
3,Edukasi,17
4,Budaya,10
5,Belanja,9
6,Kuliner,6
7,Religi,5
8,Sejarah,5
9,Petualangan,4


,final_core_label,active_count
0,Alam,114
1,Santai/Healing,90
2,Spot Foto,65
3,Outdoor,63
4,Keluarga,49
5,Ramah Anak,37
6,Gratis,36
7,Edukasi,33
8,Petualangan,22
9,Indoor,19


,metric,count
0,active_candidate,209
1,missing_category,0
2,missing_final_primary_intent,0
3,missing_final_core_labels,0
4,needs_manual_review_true,0


,metric,label_confidence
0,count,209.000000
1,mean,0.937679
2,std,0.129836
3,min,0.358000
4,5%,0.614600
5,25%,0.958000
6,50%,1.000000
7,75%,1.000000
8,95%,1.000000
9,max,1.000000


,filter,fields,supporting_active_rows,coverage_pct,readiness_status,recommendation
0,kategori wisata,"category, final_core_labels",209,100.0,strong_ready,Tampilkan sebagai filter utama
1,tujuan kunjungan,final_primary_intent,209,100.0,strong_ready,Tampilkan sebagai filter utama
2,gratis/berbayar,"price_type, price_min, price_max",209,100.0,strong_ready,Tampilkan sebagai filter utama
3,buka sekarang,"jam_buka_weekday, jam_tutup_weekday, jam_buka_weekend, jam_tutup_weekend",209,100.0,strong_ready,Tampilkan dengan guardrail jam
4,ramah anak,child_friendly_verified,66,31.6,usable_with_guardrail,Tampilkan sebagai quick filter
5,indoor,indoor_verified,33,15.8,limited_use,Tampilkan sebagai filter lanjutan
6,malam hari,night_verified,33,15.8,limited_use,Tampilkan sebagai quick filter
7,parkir,parking_verified,58,27.8,limited_use,Tampilkan sebagai filter fasilitas
8,toilet,toilet_verified,55,26.3,limited_use,Tampilkan sebagai filter fasilitas
9,mushola,mushola_verified,57,27.3,limited_use,Tampilkan sebagai filter fasilitas


,location_id,location_name,category,final_primary_intent,final_core_labels,label_confidence,needs_manual_review,review_status,final_label_source


### Keputusan: label dan filter website sudah cukup untuk versi awal

Kategori, intent, dan core label tersedia untuk active candidate. Filter utama bisa dipakai, tetapi filter berbasis fasilitas tetap harus membaca flag `true` saja dan tidak boleh menganggap nilai kosong sebagai tidak tersedia.


## Fase K - Apply Media Manual dari Workbook

Tujuan: memakai workbook media manual 39 destinasi sebagai input terkontrol. Dataset utama tidak dioverwrite; hasilnya disimpan sebagai kandidat CSV baru.


### Proses K1 - Baca Workbook Media Manual

Tujuan: membaca sheet kedua dari workbook dan mengekstrak `location_id`, nama, kategori, prioritas, dan URL foto.


In [12]:
from openpyxl import load_workbook
import re
from urllib.parse import urlparse

MEDIA_WORKBOOK_PATH = PROJECT_ROOT / "Daftar_Destinasi_Wisata_Bandung_Professional.xlsx"
MANUAL_MEDIA_EXTRACT_PATH = CURATED_DIR / "manual_media_fill_from_excel_2026-06-09.csv"
MEDIA_FILLED_OUTPUT_PATH = CURATED_DIR / "DATABASE_WISATA_LABELED_V2_REVIEWED_MEDIA_FILLED_2026-06-09.csv"
MEDIA_APPLY_SUMMARY_PATH = CURATED_DIR / "media_manual_apply_summary_2026-06-09.json"

wb = load_workbook(MEDIA_WORKBOOK_PATH, data_only=False)
ws = wb["Daftar Destinasi Wisata"]

manual_media_rows = []
for row_number in range(1, ws.max_row + 1):
    formula = ws.cell(row_number, 6).value
    location_id = ws.cell(row_number, 3).value
    if not location_id or not isinstance(formula, str) or not formula.startswith("=HYPERLINK"):
        continue
    match = re.search(r'HYPERLINK\("([^"]+)"', formula)
    media_url = match.group(1).strip() if match else ""
    manual_media_rows.append({
        "source_row": row_number,
        "priority": ws.cell(row_number, 2).value,
        "location_id": str(location_id).strip(),
        "location_name_excel": str(ws.cell(row_number, 4).value).strip(),
        "category_excel": str(ws.cell(row_number, 5).value).strip(),
        "media_image_url_candidate": media_url,
        "media_domain": urlparse(media_url).netloc if media_url else "",
        "media_source_manual": "manual_excel_2026-06-09",
        "manual_media_decision": "accepted_by_user",
    })

manual_media_df = pd.DataFrame(manual_media_rows)
display(manual_media_df.head(15))

workbook_read_summary = pd.DataFrame([
    {"metric": "workbook_exists", "value": MEDIA_WORKBOOK_PATH.exists()},
    {"metric": "sheet_name", "value": ws.title},
    {"metric": "manual_media_rows", "value": len(manual_media_df)},
    {"metric": "url_non_empty", "value": int(manual_media_df["media_image_url_candidate"].astype(str).str.strip().ne("").sum())},
    {"metric": "unique_location_id", "value": manual_media_df["location_id"].nunique()},
    {"metric": "unique_url", "value": manual_media_df["media_image_url_candidate"].nunique()},
])
display(workbook_read_summary)


,source_row,priority,location_id,location_name_excel,category_excel,media_image_url_candidate,media_domain,media_source_manual,manual_media_decision
0,6,1,LOC-017,Cihampelas Walk,Tempat Belanja,https://images.unsplash.com/photo-1567401893414-76b7b1e5a7a5?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
1,7,1,LOC-120,Rumah Guguk,Wisata Satwa,https://images.unsplash.com/photo-1548199973-03cce0bbc87b?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
2,8,1,LOC-121,NuArt Sculpture Park,Tempat Seni,https://images.unsplash.com/photo-1579783900882-c0d3dad7b119?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
3,9,1,LOC-133,Punclut Bandung (Puncak Ciumbuleuit),Wisata Alam,https://images.unsplash.com/photo-1501785888041-af3ef285b470?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
4,10,1,LOC-134,Bukit Bintang Bandung (Patahan Lembang),Wisata Alam,https://images.unsplash.com/photo-1475274047050-1d0c0975c63e?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
5,11,1,LOC-148,Pesona Nirwana Waterpark,Wahana Air,https://images.unsplash.com/photo-1519817650390-64a93db51149?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
6,12,1,LOC-173,Cakrawala Nature Sparkling Restaurant,Tempat Kuliner,https://images.unsplash.com/photo-1517248135467-4c7edcad34c4?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
7,13,1,LOC-174,Lereng Anteng Panoramic Coffee,Tempat Kuliner,https://images.unsplash.com/photo-1554118811-1e0d58224f24?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
8,14,1,LOC-177,Wahoo Waterworld,Wahana Air,https://images.unsplash.com/photo-1583417319070-4a69db38a482?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user
9,15,1,LOC-178,De'Ranch Lembang,Wisata Satwa,https://images.unsplash.com/photo-1553284965-83fd3e82fa52?w=800&q=80,images.unsplash.com,manual_excel_2026-06-09,accepted_by_user


,metric,value
0,workbook_exists,True
1,sheet_name,Daftar Destinasi Wisata
2,manual_media_rows,39
3,url_non_empty,39
4,unique_location_id,39
5,unique_url,39


#### Keputusan K1: workbook bisa dibaca sebagai input media manual

Workbook menghasilkan baris media manual yang bisa diproses. Tahap berikutnya perlu memastikan URL unik, field wajib tidak kosong, dan ID cocok dengan dataset utama.


### Proses K2 - Audit URL dan Field Wajib

Tujuan: memastikan tidak ada URL kosong, `location_id` kosong, atau URL duplikat sebelum dilakukan apply ke dataset copy.


In [13]:
manual_media_audit = pd.DataFrame([
    {"check": "rows", "count": len(manual_media_df)},
    {"check": "missing_location_id", "count": int(manual_media_df["location_id"].astype(str).str.strip().eq("").sum())},
    {"check": "missing_location_name", "count": int(manual_media_df["location_name_excel"].astype(str).str.strip().eq("").sum())},
    {"check": "missing_media_url", "count": int(manual_media_df["media_image_url_candidate"].astype(str).str.strip().eq("").sum())},
    {"check": "duplicate_location_id", "count": int(manual_media_df["location_id"].duplicated().sum())},
    {"check": "duplicate_media_url", "count": int(manual_media_df["media_image_url_candidate"].duplicated().sum())},
])
display(manual_media_audit)

domain_counts = manual_media_df["media_domain"].value_counts(dropna=False).rename_axis("media_domain").reset_index(name="count")
display(domain_counts)

duplicate_urls = manual_media_df[manual_media_df["media_image_url_candidate"].duplicated(keep=False)].sort_values("media_image_url_candidate")
display(duplicate_urls[["location_id", "location_name_excel", "media_image_url_candidate"]])


,check,count
0,rows,39
1,missing_location_id,0
2,missing_location_name,0
3,missing_media_url,0
4,duplicate_location_id,0
5,duplicate_media_url,0


,media_domain,count
0,images.unsplash.com,37
1,zumartour.com,1
2,tourbandung.com,1


,location_id,location_name_excel,media_image_url_candidate


#### Keputusan K2: URL manual layak lanjut jika tidak ada duplikat dan field wajib lengkap

Jika output menunjukkan `duplicate_media_url=0`, `missing_media_url=0`, dan `missing_location_id=0`, workbook layak dicocokkan ke dataset utama.


### Proses K3 - Cocokkan ID Workbook dengan Dataset Utama

Tujuan: memastikan semua `location_id` dari Excel ada di dataset wisata dan memang termasuk destinasi yang sebelumnya belum punya media.


In [14]:
base_media_mask = df["media_available"].astype(str).str.lower().eq("true") & df["media_image_url"].astype(str).str.strip().ne("")
base_missing_media_df = df[df["display_status"].eq("active_candidate") & ~base_media_mask].copy()

match_df = manual_media_df.merge(
    df[["location_id", "location_name", "category", "display_status", "media_available", "media_image_url", "media_audit_status"]],
    on="location_id",
    how="left",
    indicator=True,
)
match_df["name_match_exact"] = match_df["location_name_excel"].astype(str).str.strip().str.lower().eq(match_df["location_name"].astype(str).str.strip().str.lower())
match_df["was_missing_media"] = ~(
    match_df["media_available"].astype(str).str.lower().eq("true") &
    match_df["media_image_url"].astype(str).str.strip().ne("")
)

match_summary = pd.DataFrame([
    {"metric": "manual_rows", "count": len(manual_media_df)},
    {"metric": "matched_location_id", "count": int(match_df["_merge"].eq("both").sum())},
    {"metric": "unmatched_location_id", "count": int(match_df["_merge"].ne("both").sum())},
    {"metric": "name_match_exact", "count": int(match_df["name_match_exact"].sum())},
    {"metric": "active_candidate_rows", "count": int(match_df["display_status"].eq("active_candidate").sum())},
    {"metric": "was_missing_media", "count": int(match_df["was_missing_media"].sum())},
])
display(match_summary)

display(match_df[[
    "location_id", "location_name_excel", "location_name", "category_excel", "category",
    "display_status", "was_missing_media", "name_match_exact", "_merge",
]].sort_values(["_merge", "name_match_exact", "location_id"]).head(50))

unmatched_df = match_df[match_df["_merge"].ne("both")]
display(unmatched_df[["location_id", "location_name_excel", "media_image_url_candidate"]])


,metric,count
0,manual_rows,39
1,matched_location_id,39
2,unmatched_location_id,0
3,name_match_exact,39
4,active_candidate_rows,39
5,was_missing_media,39


,location_id,location_name_excel,location_name,category_excel,category,display_status,was_missing_media,name_match_exact,_merge
0,LOC-017,Cihampelas Walk,Cihampelas Walk,Tempat Belanja,Tempat Belanja,active_candidate,True,True,both
1,LOC-120,Rumah Guguk,Rumah Guguk,Wisata Satwa,Wisata Satwa,active_candidate,True,True,both
2,LOC-121,NuArt Sculpture Park,NuArt Sculpture Park,Tempat Seni,Tempat Seni,active_candidate,True,True,both
3,LOC-133,Punclut Bandung (Puncak Ciumbuleuit),Punclut Bandung (Puncak Ciumbuleuit),Wisata Alam,Wisata Alam,active_candidate,True,True,both
4,LOC-134,Bukit Bintang Bandung (Patahan Lembang),Bukit Bintang Bandung (Patahan Lembang),Wisata Alam,Wisata Alam,active_candidate,True,True,both
13,LOC-135,Taman Main Mili-Mili & Hutan Mycelia,Taman Main Mili-Mili & Hutan Mycelia,Rekreasi Keluarga,Rekreasi Keluarga,active_candidate,True,True,both
14,LOC-139,Kawah Rengganis Ciwidey,Kawah Rengganis Ciwidey,Wisata Alam,Wisata Alam,active_candidate,True,True,both
5,LOC-148,Pesona Nirwana Waterpark,Pesona Nirwana Waterpark,Wahana Air,Wahana Air,active_candidate,True,True,both
15,LOC-149,Taman Wisata Alam Cimanggu,Taman Wisata Alam Cimanggu,Wisata Alam,Wisata Alam,active_candidate,True,True,both
16,LOC-158,Situ Ninah (Situ Datar),Situ Ninah (Situ Datar),Wisata Alam,Wisata Alam,active_candidate,True,True,both


,location_id,location_name_excel,media_image_url_candidate


#### Keputusan K3: apply hanya boleh lanjut kalau semua ID match

Semua baris workbook harus cocok dengan `location_id` dataset utama. Jika ada ID tidak match, apply ditahan sampai diperbaiki.


### Proses K4 - Simulasi Apply ke Copy Dataset

Tujuan: menghitung dampak sebelum dan sesudah apply tanpa mengubah dataset utama lama.


In [15]:
apply_ready = (
    len(manual_media_df) > 0 and
    int(manual_media_audit.loc[manual_media_audit["check"].eq("missing_location_id"), "count"].iloc[0]) == 0 and
    int(manual_media_audit.loc[manual_media_audit["check"].eq("missing_media_url"), "count"].iloc[0]) == 0 and
    int(manual_media_audit.loc[manual_media_audit["check"].eq("duplicate_location_id"), "count"].iloc[0]) == 0 and
    int(manual_media_audit.loc[manual_media_audit["check"].eq("duplicate_media_url"), "count"].iloc[0]) == 0 and
    int(match_summary.loc[match_summary["metric"].eq("unmatched_location_id"), "count"].iloc[0]) == 0
)

before_df = df.copy()
after_df = df.copy()
manual_lookup = manual_media_df.set_index("location_id").to_dict(orient="index")

updated_location_ids = []
if apply_ready:
    for idx, row in after_df.iterrows():
        loc_id = row["location_id"]
        if loc_id not in manual_lookup:
            continue
        media_row = manual_lookup[loc_id]
        after_df.at[idx, "media_available"] = "True"
        after_df.at[idx, "media_image_url"] = media_row["media_image_url_candidate"]
        after_df.at[idx, "media_destination_url"] = media_row["media_image_url_candidate"]
        after_df.at[idx, "media_source"] = media_row["media_source_manual"]
        after_df.at[idx, "media_match_title"] = media_row["location_name_excel"]
        after_df.at[idx, "media_match_score"] = "1.0"
        after_df.at[idx, "media_match_method"] = "manual_excel"
        after_df.at[idx, "media_audit_status"] = "manual_accepted"
        after_df.at[idx, "media_audit_note"] = "Manual media filled from Daftar_Destinasi_Wisata_Bandung_Professional.xlsx on 2026-06-09."
        updated_location_ids.append(loc_id)

def active_media_coverage(dataframe):
    active_rows = dataframe[dataframe["display_status"].eq("active_candidate")]
    media_mask = active_rows["media_available"].astype(str).str.lower().eq("true") & active_rows["media_image_url"].astype(str).str.strip().ne("")
    return {
        "active_rows": len(active_rows),
        "media_available_rows": int(media_mask.sum()),
        "missing_media_rows": int((~media_mask).sum()),
        "coverage_pct": round(int(media_mask.sum()) / len(active_rows) * 100, 1) if len(active_rows) else 0,
    }

before_cov = active_media_coverage(before_df)
after_cov = active_media_coverage(after_df)

impact_summary = pd.DataFrame([
    {"metric": "apply_ready", "before": None, "after": apply_ready},
    {"metric": "updated_rows", "before": 0, "after": len(updated_location_ids)},
    {"metric": "active_media_available_rows", "before": before_cov["media_available_rows"], "after": after_cov["media_available_rows"]},
    {"metric": "active_missing_media_rows", "before": before_cov["missing_media_rows"], "after": after_cov["missing_media_rows"]},
    {"metric": "active_media_coverage_pct", "before": before_cov["coverage_pct"], "after": after_cov["coverage_pct"]},
])
display(impact_summary)

after_changed_preview = after_df[after_df["location_id"].isin(updated_location_ids)][[
    "location_id", "location_name", "media_available", "media_image_url", "media_source", "media_audit_status", "media_match_method"
]].copy()
display(after_changed_preview.head(50))


,metric,before,after
0,apply_ready,NaN,True
1,updated_rows,0.0,39
2,active_media_available_rows,170.0,209
3,active_missing_media_rows,39.0,0
4,active_media_coverage_pct,81.3,100.0


,location_id,location_name,media_available,media_image_url,media_source,media_audit_status,media_match_method
16,LOC-017,Cihampelas Walk,True,https://images.unsplash.com/photo-1567401893414-76b7b1e5a7a5?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
119,LOC-120,Rumah Guguk,True,https://images.unsplash.com/photo-1548199973-03cce0bbc87b?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
120,LOC-121,NuArt Sculpture Park,True,https://images.unsplash.com/photo-1579783900882-c0d3dad7b119?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
132,LOC-133,Punclut Bandung (Puncak Ciumbuleuit),True,https://images.unsplash.com/photo-1501785888041-af3ef285b470?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
133,LOC-134,Bukit Bintang Bandung (Patahan Lembang),True,https://images.unsplash.com/photo-1475274047050-1d0c0975c63e?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
134,LOC-135,Taman Main Mili-Mili & Hutan Mycelia,True,https://images.unsplash.com/photo-1448375240586-882707db888b?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
138,LOC-139,Kawah Rengganis Ciwidey,True,https://zumartour.com/wp-content/uploads/2024/11/kawah-rengganis-ciwidey-819x1024.webp,manual_excel_2026-06-09,manual_accepted,manual_excel
147,LOC-148,Pesona Nirwana Waterpark,True,https://images.unsplash.com/photo-1519817650390-64a93db51149?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
148,LOC-149,Taman Wisata Alam Cimanggu,True,https://images.unsplash.com/photo-1473448912268-2022ce9509d8?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel
157,LOC-158,Situ Ninah (Situ Datar),True,https://images.unsplash.com/photo-1504280390367-361c6d9f38f4?w=800&q=80,manual_excel_2026-06-09,manual_accepted,manual_excel


#### Keputusan K4: apply diterima jika coverage naik dan hanya baris target yang berubah

Apply manual media diterima jika `apply_ready=True`, jumlah update sama dengan jumlah workbook, dan coverage media active candidate naik sesuai target.


### Proses K5 - Simpan Output Terkontrol

Tujuan: menyimpan hasil ekstrak Excel dan dataset media-filled sebagai file baru. Dataset utama lama tetap tidak dioverwrite.


In [16]:
if not apply_ready:
    raise ValueError("Apply media manual ditahan karena audit K2/K3 belum memenuhi syarat.")

manual_media_export = manual_media_df.copy()
manual_media_export["applied_to_dataset"] = manual_media_export["location_id"].isin(updated_location_ids)
manual_media_export.to_csv(MANUAL_MEDIA_EXTRACT_PATH, index=False, encoding="utf-8-sig")
after_df.to_csv(MEDIA_FILLED_OUTPUT_PATH, index=False, encoding="utf-8-sig")

summary_payload = {
    "source_workbook": MEDIA_WORKBOOK_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "manual_media_extract_csv": MANUAL_MEDIA_EXTRACT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "media_filled_dataset_csv": MEDIA_FILLED_OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "manual_rows": int(len(manual_media_df)),
    "updated_rows": int(len(updated_location_ids)),
    "before_active_media_available_rows": int(before_cov["media_available_rows"]),
    "after_active_media_available_rows": int(after_cov["media_available_rows"]),
    "before_active_missing_media_rows": int(before_cov["missing_media_rows"]),
    "after_active_missing_media_rows": int(after_cov["missing_media_rows"]),
    "before_active_media_coverage_pct": float(before_cov["coverage_pct"]),
    "after_active_media_coverage_pct": float(after_cov["coverage_pct"]),
    "apply_ready": bool(apply_ready),
}
MEDIA_APPLY_SUMMARY_PATH.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding="utf-8")

output_files_df = pd.DataFrame([
    {"artifact": "manual_media_extract", "path": MANUAL_MEDIA_EXTRACT_PATH.relative_to(PROJECT_ROOT).as_posix(), "exists": MANUAL_MEDIA_EXTRACT_PATH.exists()},
    {"artifact": "media_filled_dataset", "path": MEDIA_FILLED_OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix(), "exists": MEDIA_FILLED_OUTPUT_PATH.exists()},
    {"artifact": "apply_summary_json", "path": MEDIA_APPLY_SUMMARY_PATH.relative_to(PROJECT_ROOT).as_posix(), "exists": MEDIA_APPLY_SUMMARY_PATH.exists()},
])
display(output_files_df)
display(pd.DataFrame([summary_payload]))


,artifact,path,exists
0,manual_media_extract,Wisata_Workspace/01_Dataset/3_Curated/manual_media_fill_from_excel_2026-06-09.csv,True
1,media_filled_dataset,Wisata_Workspace/01_Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED_MEDIA_FILLED...,True
2,apply_summary_json,Wisata_Workspace/01_Dataset/3_Curated/media_manual_apply_summary_2026-06-09.json,True


,source_workbook,manual_media_extract_csv,media_filled_dataset_csv,manual_rows,updated_rows,before_active_media_available_rows,after_active_media_available_rows,before_active_missing_media_rows,after_active_missing_media_rows,before_active_media_coverage_pct,after_active_media_coverage_pct,apply_ready
0,Daftar_Destinasi_Wisata_Bandung_Professional.xlsx,Wisata_Workspace/01_Dataset/3_Curated/manual_media_fill_from_excel_2026-06-09.csv,Wisata_Workspace/01_Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED_MEDIA_FILLED...,39,39,170,209,39,0,81.3,100.0,True


#### Keputusan K5: dataset media-filled menjadi kandidat baru

Output baru dipakai sebagai kandidat dataset wisata dengan media terisi. File lama tetap menjadi baseline pembanding sampai validation tahap berikutnya selesai.


## Ringkasan Sementara Setelah Fase K

| Area | Keputusan |
|---|---|
| Workbook media manual | Dipakai sebagai input terkontrol |
| Dataset utama lama | Tidak dioverwrite |
| Output baru | Dataset media-filled CSV |
| Media coverage | Divalidasi lewat perbandingan sebelum/sesudah |
| Tahap berikutnya | Fase L: manual completion queue atau validation lanjutan |

Fase K menyelesaikan missing media berdasarkan workbook manual dan menjaga lineage file tetap jelas.


## Fase L - Runtime Dataset Candidate

Tahap ini hanya mengunci arah dataset runtime. Tidak perlu code karena ini keputusan kerja, bukan perhitungan baru.

**Keputusan:** dataset kandidat runtime wisata memakai `DATABASE_WISATA_LABELED_V2_REVIEWED_MEDIA_SENTIMENT_RUNTIME_CANDIDATE_2026-06-09.csv`. File lama tetap menjadi checkpoint, jadi pipeline tidak bolak-balik membaca versi yang berbeda.

## Fase M - Contract Sentiment Fallback

Tahap ini mencatat aturan sumber sentiment yang boleh dipakai sistem.

**Keputusan:** `tfidf_linearsvc` tetap sumber utama NLP. `google_maps_stars_fallback` diterima sebagai fallback dari review Google Maps yang sudah dicek, tetapi harus tetap dibedakan dari model NLP. Jika sentiment kosong, sistem harus menurunkan confidence, bukan membuat klaim terlalu yakin.

## Fase N - Audit Input Scoring Recommender

Tujuan kecil tahap ini adalah memastikan komponen scoring sudah tersedia sebelum recommender diuji dengan query user. Cell ini belum melakukan tuning ranking; hanya mengecek bahan yang akan dipakai ranking.

In [ ]:
from pathlib import Path
import pandas as pd

DATASET_PATH = Path("Wisata_Workspace/01_Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED_MEDIA_SENTIMENT_RUNTIME_CANDIDATE_2026-06-09.csv")

df = pd.read_csv(DATASET_PATH)
active = df[df["display_status"].astype(str).str.lower().eq("active_candidate")].copy()

def bool_true(series):
    return series.astype(str).str.lower().eq("true")

def confidence_label(count):
    try:
        count = float(count)
    except Exception:
        count = 0.0
    if count <= 0:
        return "unavailable"
    if count < 30:
        return "low_review_confidence"
    if count < 100:
        return "medium_review_confidence"
    return "high_review_confidence"

summary = pd.DataFrame([
    ("rows_total", len(df)),
    ("active_candidate", len(active)),
    ("media_available_active", int(bool_true(active["media_available"]).sum())),
    ("sentiment_available_active", int(bool_true(active["sentiment_available"]).sum())),
    ("coordinate_verified_active", int(bool_true(active["coordinate_verified"]).sum())),
    ("price_verified_active", int(bool_true(active["price_verified"]).sum())),
], columns=["metric", "value"])

source_counts = (
    active["sentiment_model_source"]
    .fillna("")
    .replace("", "blank")
    .value_counts()
    .reset_index()
)
source_counts.columns = ["sentiment_model_source", "active_rows"]

active["review_confidence_label_runtime"] = active["total_ulasan"].apply(confidence_label)
confidence_counts = active["review_confidence_label_runtime"].value_counts().reset_index()
confidence_counts.columns = ["review_confidence_label_runtime", "active_rows"]

missing = active[~bool_true(active["sentiment_available"])][["location_id", "location_name"]]

print("=== RUNTIME CANDIDATE SCORING INPUT AUDIT ===")
print(summary.to_string(index=False))
print("\n=== SENTIMENT SOURCE COUNTS ===")
print(source_counts.to_string(index=False))
print("\n=== REVIEW CONFIDENCE LABEL COUNTS (runtime estimate from total_ulasan) ===")
print(confidence_counts.to_string(index=False))
print("\n=== REMAINING SENTIMENT MISSING ===")
print(missing.to_string(index=False) if len(missing) else "Tidak ada active candidate yang sentiment-nya kosong.")

=== RUNTIME CANDIDATE SCORING INPUT AUDIT ===
                    metric  value
                rows_total    234
          active_candidate    209
    media_available_active    209
sentiment_available_active    207
coordinate_verified_active    200
     price_verified_active    209

=== SENTIMENT SOURCE COUNTS ===
    sentiment_model_source  active_rows
           tfidf_linearsvc          193
google_maps_stars_fallback           14
               unavailable            2

=== REVIEW CONFIDENCE LABEL COUNTS (runtime estimate from total_ulasan) ===
review_confidence_label_runtime  active_rows
         high_review_confidence           89
       medium_review_confidence           85
          low_review_confidence           33
                    unavailable            2

=== REMAINING SENTIMENT MISSING ===
location_id         location_name
    LOC-196 Curug Walanda Citatah
    LOC-213      Taman Cibeunying


**Keputusan Fase N**

Dataset runtime candidate sudah layak dipakai untuk audit rekomendasi: media dan harga aktif sudah penuh, sentiment aktif hampir penuh, dan sisa kosong tinggal dua lokasi. Tahap berikutnya adalah mengarahkan runtime/app ke dataset ini, lalu menjalankan query uji untuk melihat apakah ranking sudah masuk akal.